<a href="https://colab.research.google.com/github/JAVERIAADIL/Learning-GPU-infrastructure/blob/module1/VectorAdd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
from numba import cuda
import time

In [14]:
@cuda.jit # it is basically to launch a kernal in python language underneath there is cuda line __global__
def vectorAdd(a,b,c, n):
  i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x #here basically we are getting each value that is assigned to each thread in its block
  if (i < n): # since we are taking 100 numbers and each block conatin 32 threads so we need 4 block total but 101 to 128 thread have no value so we define this condition
    c[i] = a[i] + b[i]

In [23]:
n = 100000000
a = np.random.randn(n).astype(np.float32)
b = np.random.randn(n).astype(np.float32)
c = np.zeros(n, dtype = np.float32)

In [24]:
d_a = cuda.to_device (a) #it basically copy vector from cpu to gpu
d_b = cuda.to_device (b)
d_c = cuda.to_device (c)


In [25]:
threads = 32
block = ( n + threads - 1) // threads

In [26]:
start_event = cuda.event() #we donot use time here as cuda.event lives on gpu and measure gpu work accurately
end_event = cuda.event()

start_event.record()
vectorAdd[block, threads](d_a,d_b, d_c,n)
end_event.record()
end_event.synchronize() #gpu  works asynchronize so in order to note time we have to synchronize it so it wait for gpu to finish then note time
print ( f" GPU Average: {cuda.event_elapsed_time(start_event, end_event):.3f} ms") # here we are measuring time of average since there would be so much values if we take single time individual

 GPU Average: 11.529 ms


In [27]:
c_gpu = d_c.copy_to_host()
#print(c_gpu)

In [28]:
start = time.perf_counter() # perf_counter is high measured CPU clock even accurately measure microseconds
for i in range(n):
  c[i] = a[i] + b[i]
end = time.perf_counter()
#print(c)
print ( f" CPU  Loop Time Average: {(end-start)* 1000 :.3f} ms")

 CPU  Loop Time Average: 47103.321 ms


In [29]:
start = time.perf_counter()
c_numpy = a + b
end = time.perf_counter()
#print(c_numpy)
print ( f" CPU Numpy Time Average: {(end-start)* 1000 :.3f} ms")

 CPU Numpy Time Average: 152.299 ms


In [22]:
print(f"Results match: {np.allclose(c_gpu, c, c_numpy)}")

Results match: True
